# EXP027: MFI + EAP with uniform prior U(-4, 4) (Python)

`EXP027/Test_MFI_EAP_unif_on_the_simulated_bank.R` の Python notebook 版です。DQN と MFI を極力同条件で比較するため、`Train_and_test_DQN_on_the_simulated_banks.ipynb` と同じ `RESPOND`、`FI`、`EAP_quadrature`（62点、U(-4, 4)）を使用します。

被験者を step ごとに一括処理する乱数生成順も DQN の `TEST` に揃えています。各 step では、現在の EAP 推定値における Fisher 情報量が最大の未出題項目を選択します。既存の catR 版結果を上書きしないよう、出力ファイル名には既定で `_python` を付けます。

In [3]:
# -*- coding: utf-8 -*-
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple

import numpy as np
import pandas as pd


def find_project_root():
    candidates = []
    if "__file__" in globals():
        script_dir = Path(__file__).resolve().parent
        candidates.extend([script_dir, *script_dir.parents])

    cwd = Path.cwd().resolve()
    candidates.extend(
        [
            cwd,
            *cwd.parents,
            cwd / "Grad_Research",
            cwd
            / "Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History",
            Path("/content/Grad_Research"),
            Path(
                "/content/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
            Path("/content/drive/MyDrive/Colab Notebooks/Grad_Research"),
            Path(
                "/content/drive/MyDrive/Colab Notebooks/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History"
            ),
        ]
    )

    for root in candidates:
        if (root / "data").is_dir():
            return root

    try:
        from google.colab import drive

        drive.mount("/content/drive")
    except Exception:
        pass

    for root in candidates:
        if (root / "data").is_dir():
            return root

    raise FileNotFoundError(
        "Could not find the project root. "
        "In Colab, place the repository at MyDrive/Grad_Research or /content/Grad_Research."
    )


ROOT = find_project_root()
RESULTS_DIR = ROOT / "EXP027" / "results"

print(f"Project root: {ROOT}")
print(f"Results dir : {RESULTS_DIR}")

Project root: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History
Results dir : /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP027/results


In [4]:
@dataclass
class Config:
    test_length: int = 40

    # EAP quadrature (uniform prior U(prior_low, prior_high))
    n_quad: int = 62
    prior_low: float = -4.0
    prior_high: float = 4.0

    # Bank / evaluation data
    bank_type: str = "uncor"  # 'uncor' | 'cor'
    bank_id: int = 1
    n_items: int = 500
    testing_size: int = 0  # 0: use all theta values
    theta_csv: str = ""  # empty: data/theta_true/theta_true_{bank_id}.csv

    # Reproducibility / output
    seed: int = 20260430
    output_suffix: str = "_python"

In [5]:
# The following three functions are aligned with the EXP027 DQN notebook.
def RESPOND(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    p = (1 - c) / (1 + np.exp(-D * a * (theta - b))) + c
    return (np.random.random(size=p.shape) <= p).astype(int)


def FI(item_para, theta, D=1):
    a = item_para[:, 0]
    b = item_para[:, 1]
    c = item_para[:, 2]
    return (
        D**2
        * a**2
        * (1 - c)
        / (c + np.exp(D * a * (theta - b)))
        / (1 + np.exp(-D * a * (theta - b))) ** 2
    )


def EAP_quadrature(
    item_paras: np.ndarray,
    resp: np.ndarray,
    n_quad: int = 62,
    prior_low: float = -4.0,
    prior_high: float = 4.0,
    D: float = 1.0,
) -> Tuple[float, float]:
    """EAP mean and posterior variance via quadrature with uniform prior U(prior_low, prior_high).

    Parameters
    ----------
    item_paras : (n_items, 3) array  [a, b, c]
    resp       : (n_items,)  array  0/1

    Returns
    -------
    (eap_mean, eap_variance)
    """
    theta_grid = np.linspace(prior_low, prior_high, n_quad)
    # Uniform prior: constant log-density over the grid, cancels after normalization
    log_prior = np.zeros(n_quad)

    a = item_paras[:, 0]
    b = item_paras[:, 1]
    c = item_paras[:, 2]
    # shape: (n_items, n_quad)
    p = c[:, None] + (1 - c[:, None]) / (
        1 + np.exp(-D * a[:, None] * (theta_grid[None, :] - b[:, None]))
    )
    p = np.clip(p, 1e-10, 1 - 1e-10)
    log_lik = np.sum(
        resp[:, None] * np.log(p) + (1 - resp[:, None]) * np.log(1 - p),
        axis=0,
    )  # shape: (n_quad,)

    log_post = log_lik + log_prior
    log_post -= log_post.max()
    post = np.exp(log_post)
    post /= post.sum()

    eap_mean = float(np.sum(theta_grid * post))
    eap_var = float(np.sum((theta_grid - eap_mean) ** 2 * post))
    return eap_mean, eap_var

In [6]:
def choose_mfi(item_bank, theta_current, item_ids):
    """Choose each subject's unadministered item with maximum FI."""
    information = np.vstack([FI(item_bank, theta) for theta in theta_current])

    if item_ids.shape[0] > 0:
        subject_indices = np.arange(len(theta_current))[:, None]
        information[subject_indices, item_ids.T] = -np.inf

    return information.argmax(axis=1).astype(np.int64)


def summarize_steps(theta_true, theta_history):
    rows = []
    theta_true_sd = np.std(theta_true, ddof=1)

    for step, theta_est in enumerate(theta_history, start=1):
        bias = theta_est - theta_true
        theta_est_sd = np.std(theta_est, ddof=1)
        correlation = (
            np.nan
            if theta_true_sd == 0 or theta_est_sd == 0
            else np.corrcoef(theta_true, theta_est)[0, 1]
        )
        rows.append(
            {
                "step": step,
                "Bias": np.mean(bias),
                "RMSE": np.sqrt(np.mean(bias**2)),
                "MAE": np.mean(np.abs(bias)),
                "r": correlation,
            }
        )

    return pd.DataFrame(rows)


def run_mfi_cat(cfg, item_bank, theta_true):
    # Use the same NumPy RNG and subject-vectorized order as DQN TEST.
    np.random.seed(cfg.seed)
    testing_size = len(theta_true)

    theta_current = np.random.rand(testing_size) - 0.5
    item_ids = np.empty((0, testing_size), dtype=np.int64)
    responses = np.empty((0, testing_size), dtype=np.int64)
    theta_history = np.empty((0, testing_size), dtype=float)

    for step in range(cfg.test_length):
        selected = choose_mfi(item_bank, theta_current, item_ids)
        step_responses = RESPOND(item_bank[selected], theta_true)

        item_ids = np.concatenate((item_ids, selected[np.newaxis, :]))
        responses = np.concatenate((responses, step_responses[np.newaxis, :]))

        theta_current = np.zeros(testing_size)
        for subject in range(testing_size):
            theta_current[subject], _ = EAP_quadrature(
                item_bank[item_ids[:, subject]],
                responses[:, subject],
                n_quad=cfg.n_quad,
                prior_low=cfg.prior_low,
                prior_high=cfg.prior_high,
            )

        theta_history = np.concatenate((theta_history, theta_current[np.newaxis, :]))
        bias = theta_current - theta_true
        print(
            "step {:g}, bias {:.3f}, rmse {:.3f}, mae {:.3f}".format(
                step + 1,
                np.mean(bias),
                np.sqrt(np.mean(bias**2)),
                np.mean(np.abs(bias)),
            )
        )

    user_id_col = np.repeat(np.arange(1, testing_size + 1), cfg.test_length)
    step_col = np.tile(np.arange(1, cfg.test_length + 1), testing_size)
    records = pd.DataFrame(
        {
            "userID": user_id_col,
            "step": step_col,
            "itemID": (item_ids + 1).T.reshape(-1),
            "resp": responses.T.reshape(-1),
            "theta_true": np.repeat(theta_true, cfg.test_length),
            "theta_est": theta_history.T.reshape(-1),
            "bias": (theta_history - theta_true).T.reshape(-1),
        }
    )
    summary_by_step = summarize_steps(theta_true, theta_history)
    return records, summary_by_step

In [7]:
cfg = Config(
    test_length=40,
    n_quad=62,
    prior_low=-4.0,
    prior_high=4.0,
    bank_type="uncor",
    bank_id=1,
    n_items=500,
    testing_size=0,
    theta_csv="",
    seed=20260430,
    output_suffix="_python",
)

bank_dir = {
    "uncor": ROOT / "data" / "uncorrelated_banks",
    "cor": ROOT / "data" / "correlated_banks",
}.get(cfg.bank_type)
if bank_dir is None:
    raise ValueError("bank_type must be 'uncor' or 'cor'.")

bank_path = bank_dir / f"item_bank_{cfg.bank_type}_{cfg.bank_id}.csv"
item_bank = pd.read_csv(bank_path)[["a", "b", "c"]].to_numpy()[: cfg.n_items]

if cfg.theta_csv:
    theta_path = Path(cfg.theta_csv).expanduser()
    if not theta_path.is_absolute():
        theta_path = ROOT / theta_path
else:
    theta_path = ROOT / "data" / "theta_true" / f"theta_true_{cfg.bank_id}.csv"

theta_true = pd.read_csv(theta_path)["x"].to_numpy()
if not cfg.theta_csv and cfg.testing_size > 0:
    theta_true = theta_true[: cfg.testing_size]

if cfg.test_length > len(item_bank):
    raise ValueError("test_length cannot exceed the number of items in the bank.")
if len(theta_true) < 2:
    raise ValueError("At least two theta values are required to calculate correlation.")

print(f"item bank  : {item_bank.shape} ({bank_path})")
print(f"theta_true : {theta_true.shape} ({theta_path})")
print(f"Config     : {cfg}")

item bank  : (500, 3) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/data/uncorrelated_banks/item_bank_uncor_1.csv)
theta_true : (5000,) (/Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/data/theta_true/theta_true_1.csv)
Config     : Config(test_length=40, n_quad=62, prior_low=-4.0, prior_high=4.0, bank_type='uncor', bank_id=1, n_items=500, testing_size=0, theta_csv='', seed=20260430, output_suffix='_python')


In [8]:
records, summary_by_step = run_mfi_cat(cfg, item_bank, theta_true)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
stem = f"{cfg.bank_type}_{cfg.bank_id}_MFI_EAP_unif{cfg.output_suffix}"
records_path = RESULTS_DIR / f"records_{stem}.csv"
summary_path = RESULTS_DIR / f"summary_{stem}.csv"
records.to_csv(records_path, index=False)
summary_by_step.to_csv(summary_path, index=False)

display(summary_by_step.tail(1))
print(f"Saved records to: {records_path}")
print(f"Saved summary to: {summary_path}")

step 1, bias 0.064, rmse 1.340, mae 1.115
step 2, bias -0.128, rmse 1.314, mae 1.105
step 3, bias -0.117, rmse 1.129, mae 0.890
step 4, bias -0.092, rmse 1.031, mae 0.816
step 5, bias -0.083, rmse 0.952, mae 0.746
step 6, bias -0.085, rmse 0.874, mae 0.685
step 7, bias -0.080, rmse 0.806, mae 0.630
step 8, bias -0.077, rmse 0.750, mae 0.582
step 9, bias -0.068, rmse 0.701, mae 0.543
step 10, bias -0.061, rmse 0.662, mae 0.515
step 11, bias -0.052, rmse 0.624, mae 0.487
step 12, bias -0.049, rmse 0.599, mae 0.465
step 13, bias -0.044, rmse 0.572, mae 0.447
step 14, bias -0.042, rmse 0.543, mae 0.425
step 15, bias -0.038, rmse 0.519, mae 0.406
step 16, bias -0.033, rmse 0.500, mae 0.392
step 17, bias -0.034, rmse 0.478, mae 0.374
step 18, bias -0.033, rmse 0.462, mae 0.363
step 19, bias -0.035, rmse 0.448, mae 0.352
step 20, bias -0.028, rmse 0.434, mae 0.342
step 21, bias -0.026, rmse 0.419, mae 0.332
step 22, bias -0.023, rmse 0.409, mae 0.323
step 23, bias -0.022, rmse 0.395, mae 0.31

,step,Bias,RMSE,MAE,r
39,40,-0.015104,0.294926,0.232626,0.961065


Saved records to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP027/results/records_uncor_1_MFI_EAP_unif_python.csv
Saved summary to: /Users/itsuki/Adaptive-Testing-Based-on-Reinforcement-Learning-Considering-Response-History/EXP027/results/summary_uncor_1_MFI_EAP_unif_python.csv
